# GLM-4-Flash 通用能力评测数据分析

本 notebook 基于 Pandas 对 `results/glm4_results.csv` 评测结果进行清洗、统计与可视化分析，作为 Pandas 数据分析能力的实践教材，同时验证报告中的统计数字。

**数据来源**：`results/glm4_results.csv`（由 `run_eval.py` 评测生成）与 `data/eval_questions.csv`（题集）。

## 一、数据加载

> 说明：Jupyter 默认工作目录是 notebook 所在目录，所以相对路径（如 `results/glm4_results.csv`）直接可用，无需拼接绝对路径。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 统一设置中文字体，避免图表中文乱码
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 读取评测结果与题集（utf-8-sig 兼容 BOM 头）
df = pd.read_csv('results/glm4_results.csv', encoding='utf-8-sig')
qs = pd.read_csv('data/eval_questions.csv', encoding='utf-8-sig')
print(f'评测结果 {df.shape}，题集 {qs.shape}')
df.head()

## 二、数据概览与清洗检查

检查缺失值与重复行，体现数据清洗意识（本次数据已规整，预期无缺失无重复）。

In [ ]:
# 基本信息：列名、数据类型、非空计数
df.info()
print('\n--- 缺失值检查 ---')
print(df.isnull().sum())
print(f'\n重复行数: {df.duplicated().sum()}')
print('\n--- correct 列取值分布 ---')
print(df['correct'].value_counts())

## 三、总体准确率

将 `correct` 列（Y/N）转为布尔后求均值，即为总体准确率。

In [ ]:
# correct=Y 记 1，N 记 0，均值即准确率
total = len(df)
right = (df['correct'] == 'Y').sum()
acc = right / total * 100
parse_fail = (df['model_answer'] == 'PARSE_FAIL').sum()
print(f'总题数: {total}')
print(f'正确数: {right}')
print(f'总体准确率: {acc:.1f}%')
print(f'格式解析失败: {parse_fail} 题')

## 四、分学科准确率

按 `subject` 分组，统计每学科正确数、总数与准确率，与报告数字对照验证。

In [ ]:
# 按 subject 分组：正确数 = correct=='Y' 的和，总数 = 学科题数
by_subj = df.groupby('subject').agg(
    正确数=('correct', lambda s: (s == 'Y').sum()),
    总数=('correct', 'count')
)
by_subj['准确率(%)'] = (by_subj['正确数'] / by_subj['总数'] * 100).round(1)
# 按准确率升序排列，便于识别薄弱学科
by_subj = by_subj.sort_values('准确率(%)')
by_subj

## 五、错题分析

筛选错题并查看模型答案与标准答案的对照，定位薄弱知识点。

In [ ]:
# 筛选错题：correct != Y
wrongs = df[df['correct'] != 'Y'][['idx', 'subject', 'question', 'std_answer', 'model_answer']]
print(f'错题数: {len(wrongs)}')
wrongs

## 六、交叉分析：学科 × 对错

用 `pivot_table` 构建学科 × 对错交叉表，直观呈现各学科对错分布。

In [ ]:
# pivot_table：行=学科，列=correct，值=题目计数
pivot = pd.pivot_table(df, index='subject', columns='correct', values='idx', aggfunc='count', fill_value=0)
pivot.columns = ['错误(N)' if c == 'N' else '正确(Y)' for c in pivot.columns]
pivot['总数'] = pivot.sum(axis=1)
pivot['准确率(%)'] = (pivot.get('正确(Y)', 0) / pivot['总数'] * 100).round(1)
pivot

## 七、可视化

比分学科报告图表更精细：配色渐变、网格线、柱顶数值标注、规范标题。

In [ ]:
# 分学科准确率柱状图：柱顶标注百分比，附总体参考线
fig, ax = plt.subplots(figsize=(9, 5))
subjects = by_subj.index.tolist()
accs = by_subj['准确率(%)'].tolist()
# 按准确率高低渐变配色，薄弱学科偏红
colors = ['#E74C3C' if a < 95 else '#4C9EEB' for a in accs]
bars = ax.bar(subjects, accs, color=colors, edgecolor='white', linewidth=1.2)
# 柱顶数值标注
for b, a in zip(bars, accs):
    ax.text(b.get_x() + b.get_width()/2, a + 1, f'{a:.1f}%', ha='center', fontsize=10, fontweight='bold')
# 总体准确率参考线
ax.axhline(acc, color='#9B59B6', linestyle='--', linewidth=1.5, label=f'总体准确率 {acc:.1f}%')
ax.set_ylim(0, 115)
ax.set_ylabel('准确率(%)')
ax.set_title('GLM-4-Flash 分学科准确率（薄弱学科标红）')
ax.grid(axis='y', linestyle=':', alpha=0.6)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# 正确/错误占比饼图
fig, ax = plt.subplots(figsize=(5, 5))
counts = [(df['correct']=='Y').sum(), (df['correct']!='Y').sum()]
labels = [f'正确 ({counts[0]})', f'错误 ({counts[1]})']
ax.pie(counts, labels=labels, autopct='%1.1f%%', colors=['#7ED321', '#E74C3C'], startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax.set_title(f'总体作答结果分布（共 {len(df)} 题）')
plt.tight_layout()
plt.show()

## 八、分析结论

基于以上数据推导的真实结论：

1. **总体表现优秀但非满分**：60 题答对 57 题，准确率 95.0%，格式解析失败 0 题，说明 Prompt 约束（只输出字母）生效，答案提取稳定。
2. **最薄弱学科为数学/语文/地理（均 90%）**：三学科各错 1 题。其中数学为推理错误（圆面积误用周长公式），语文与地理为知识性错误（古汉语被动句、省级行政区接壤），反映模型在事实性记忆与公式适用边界上仍有提升空间。
3. **答案提取稳定性高**：0 题解析失败，多级提取策略（明确→隐含→回退）对 GLM-4-Flash 的回答格式适配良好。
4. **学科间差异有限**：最高 100%（计算机/历史/法律）与最低 90% 仅差 10 个百分点，模型能力分布较均衡，无明显短板学科。
5. **样本量局限**：每学科仅 10 题，准确率波动较大（错 1 题即下降 10 个百分点），结论仅作趋势参考，需更大样本验证稳定性。